Imports:

In [2]:
import gzip
import os
import re
from gensim.models import Word2Vec

### We used the dutch OpenSubtitles Corpus as it matches our goal for TV emotion classification

OpenSubtitles Dutch Corpus (OPUS Collection)
Source: Movie and TV subtitle translations from OpenSubtitles.org, compiled by the OPUS project NlplPapers with Code

https://opus.nlpl.eu/OpenSubtitles/nl&en/v2024/OpenSubtitles

https://www.opensubtitles.org/en/search/subs


Scale: Part of collection spanning 60 corpora in 58 languages with 2.6 billion sentences total OpenSubtitles parallel corpora

Domain: Movies and television shows (conversational dialogue)

Language variety:  Dutch

Text type: Subtitle transcripts representing natural dialogue and speech patterns

Countries of origin: Dutch-language films and international content dubbed/subtitled in Dutch

Temporal range: Various decades of film/TV content up to recent releases

Data structure: Monolingual Dutch sentences extracted from subtitle files



In [24]:
def extract_and_prepare_corpus(gz_file_path, max_sentences=1_000_000):
    """Extract corpus, show statistics, and prepare for training"""
    
    # Extract the file
    output_file = gz_file_path.replace('.gz', '')
    
    with gzip.open(gz_file_path, 'rt', encoding='utf-8') as f_in:
        with open(output_file, 'w', encoding='utf-8') as f_out:
            f_out.write(f_in.read())
    
    sentences = []
    line_lengths = []
    
    print("First 10 lines:")
    with open(output_file, 'r', encoding='utf-8') as f:
        for line_num, line in enumerate(f):
            line = line.strip()
            
            # Show first 10 lines
            if line_num < 10:
                print(f"{line_num+1}: {line}")
            
            if line:
                words = line.split()
                if len(words) >= 1:
                    sentences.append(words)
                    line_lengths.append(len(words))
                
                # Stop at max for training
                if len(sentences) >= max_sentences:
                    break
    
    # Show statistics
    total_processed = line_num + 1
    print(f"\nDataset Statistics:")
    print(f"Total lines processed: {total_processed:,}")
    print(f"Sentences for training: {len(sentences):,}")
    print(f"Average tokens per sentence: {sum(line_lengths)/len(line_lengths):.1f}")
    print(f"Min/Max tokens: {min(line_lengths)}/{max(line_lengths)}")
    
    return sentences, output_file

In [25]:
training_sentences, corpus_file = extract_and_prepare_corpus("nl.tok.gz")


First 10 lines:
1: WERKKAMP CHANGZHOU JIANGSU , CHINA
2: VACCINATIEPROGRAMMA WERELDGEZONDHEIDSORGANISATIE
3: Wat is er ?
4: Wat is er mis ?
5: - Ga weg bij haar .
6: Blijf uit haar buurt .
7: Bel een ambulance .
8: Ze heeft een hartstilstand .
9: - Ze kan niet weg .
10: We gaan naar het ziekenhuis .

Dataset Statistics:
Total lines processed: 1,000,000
Sentences for training: 1,000,000
Average tokens per sentence: 7.1
Min/Max tokens: 1/86


### training

In [26]:
def train_and_evaluate_model(sentences, params, name):
    """Train model with specific parameters and evaluate"""
    print(f"\n=== Training {name} ===")
    print(f"Parameters: {params}")
    
    model = Word2Vec(sentences=sentences, **params)
    
    # Quick evaluation
    test_words = ['goed', 'slecht', 'mooi', 'blij']
    print(f"Vocabulary size: {len(model.wv.index_to_key)}")
    
    for word in test_words:
        if word in model.wv:
            similar = model.wv.most_similar(word, topn=3)
            print(f"'{word}': {[w for w, score in similar]}")
    
    return model

# Test different configurations
configs = {
    "Baseline": {"vector_size": 200, "window": 5, "min_count": 5, "sg": 1, "epochs": 10, "workers": 8},
    "larger_vector_size": {"vector_size": 500, "window": 5, "min_count": 5, "sg": 1, "epochs": 10, "workers": 8},
    "smaller_window": {"vector_size": 200, "window": 3, "min_count": 5, "sg": 1, "epochs": 10, "workers": 8},
    "higher_mincount": {"vector_size": 200, "window": 5, "min_count": 10, "sg": 1, "epochs": 10, "workers": 8},
    "cbow_model": {"vector_size": 200, "window": 5, "min_count": 5, "sg": 0, "epochs": 10, "workers": 8},
    "Higher_epochs": {"vector_size": 200, "window": 5, "min_count": 5, "sg": 0, "epochs": 20, "workers": 8}
}

models = {}
for name, params in configs.items():
    models[name] = train_and_evaluate_model(training_sentences, params, name)


=== Training Baseline ===
Parameters: {'vector_size': 200, 'window': 5, 'min_count': 5, 'sg': 1, 'epochs': 10, 'workers': 8}
Vocabulary size: 30549
'goed': ['slecht', 'Slecht', 'amusant']
'slecht': ['goed', 'chic', 'Slecht']
'mooi': ['schitterend', 'fraai', 'prachtig']
'blij': ['Blij', 'opgelucht', 'Fijn']

=== Training larger_vector_size ===
Parameters: {'vector_size': 500, 'window': 5, 'min_count': 5, 'sg': 1, 'epochs': 10, 'workers': 8}
Vocabulary size: 30549
'goed': ['chic', 'amusant', 'begrijpelijk']
'slecht': ['goed', 'ongunstig', 'gehumeurd']
'mooi': ['fraai', 'boeiend', 'schitterend']
'blij': ['opgelucht', 'Blij', 'opgetogen']

=== Training smaller_window ===
Parameters: {'vector_size': 200, 'window': 3, 'min_count': 5, 'sg': 1, 'epochs': 10, 'workers': 8}
Vocabulary size: 30549
'goed': ['chic', 'amusant', 'slecht']
'slecht': ['goed', 'geweldig', 'schokkend']
'mooi': ['prachtig', 'fraai', 'leuk']
'blij': ['opgelucht', 'Blij', 'opgetogen']

=== Training higher_mincount ===
Para

Parameter Justification:
- vector_size = 500: Balance between expressiveness and efficiency and no longer had the goed/slecht pairing
- window = 5: Captures sufficient conversational context
- min_count = 5: Filters noise, reduces vocab from 52K to 30K words
- sg = 1 (Skip-gram): Better for rare emotional vocabulary than CBOW
- epochs = 10: Sufficient convergence without overfitting

In [ ]:
print(f"Using {len(training_sentences)} sentences for training")
model = Word2Vec(
    sentences=training_sentences,
    vector_size=500,
    window=5,
    min_count=5,
    workers=4,
    epochs=10,
    sg=1,
    seed=42
)

model.save("dutch_custom_embeddings_final.model")

print(f"Training completed with {len(model.wv.index_to_key)} vocabulary words")

Using 1000000 sentences for training
Training completed with 19311 vocabulary words


### Observed Results:
- Successfully learned semantic clusters (e.g., "mooi" with "prachtig", "fraai")
- Identified challenge: antonyms co-occur in dialogue
